In [1]:
import torch
from torch.cuda import device

'gpu' if torch.cuda.is_available() else 'cpu'

'gpu'

In [2]:
import torch

import math

def attention(q,k,v,mask=None):
    k=k.transpose(-2,-1)
    d_k=q.size(-1)
    z=torch.matmul(q,k)/math.sqrt(d_k)
    
    if mask is not None:
        z=z.masked_fill(mask==0,-1e9)
        
    weights=z.softmax(dim=-1)     
    
    return z, weights, torch.matmul(weights,v)
    

In [3]:
seq_len=5
k=torch.randn(size=(seq_len,8))
v=torch.randn(size=(seq_len,8))
q=torch.randn(size=(seq_len,8))

print(f"k is:{k}")
print(f"v is:{v}")
print(f"q is:{q}")

k is:tensor([[-0.6093,  0.4726, -0.6798,  0.4405, -2.7561,  1.0314,  0.7279,  0.5535],
        [-0.0134, -0.1641,  0.8463, -1.2417, -0.0998,  0.4888, -0.9428, -0.5910],
        [-0.6844, -2.1110, -1.6378,  0.4182,  0.1911,  1.0338,  0.0092,  1.0991],
        [-0.2384,  1.3470, -0.0872, -1.4418, -0.1324,  0.5528, -0.7967, -0.3874],
        [ 1.0022, -0.2827, -2.0059,  1.3234,  0.3679,  0.1031, -1.2134, -1.5299]])
v is:tensor([[-0.9870, -0.1056,  0.7067, -0.0642,  0.2831,  1.5207,  0.7967,  0.1624],
        [ 0.0831, -0.2381, -1.1929,  0.4566, -0.1978,  0.9431,  1.8090, -0.0092],
        [ 0.1725,  1.2960, -0.9284, -0.2940, -0.3037, -0.1353, -0.4350, -0.0793],
        [-0.1711,  0.6942,  0.0894, -0.6206, -0.6741, -0.2852,  0.4580,  1.3586],
        [ 1.0273,  0.2664,  0.2544,  0.4543, -0.0794,  1.8079,  0.2161,  0.6941]])
q is:tensor([[ 3.1440e-01,  7.7442e-01,  7.4985e-01, -2.3282e+00, -9.1093e-01,
         -5.9602e-01, -1.5406e-03, -2.4394e+00],
        [-1.7103e+00, -1.1013e+00,  2.22

In [4]:
z, weights, attention_score=attention(q,k,v,)
print(f"attention score is={attention_score}")

attention score is=tensor([[-0.0494,  0.2421, -0.3995, -0.0837, -0.3864,  0.4423,  1.0178,  0.6791],
        [-0.3580,  0.4946, -0.0266, -0.1204,  0.0035,  0.8385,  0.2712,  0.1066],
        [ 0.0217,  0.4481, -0.2689, -0.0498, -0.2180,  0.6691,  0.5148,  0.4130],
        [-0.1080,  0.3951, -0.5365, -0.0632, -0.2617,  0.4737,  0.7580,  0.3165],
        [-0.6962,  0.1728,  0.3510, -0.1015,  0.1338,  1.1662,  0.5752,  0.1614]])


In [5]:
print(torch.sum(weights,axis=-1))

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [6]:
print(z)

tensor([[-0.2887,  1.6394, -2.6599,  1.7666, -0.4072],
        [ 1.7191, -1.6710,  1.5112, -2.2827, -0.3231],
        [-0.4105, -0.3447, -0.0877, -0.2480, -0.4550],
        [ 0.2480,  1.1101,  0.7894,  0.6718, -3.0821],
        [ 2.9106, -0.7474,  1.4711, -0.3317, -0.7519]])


In [7]:
x=torch.rand(size=(seq_len,seq_len))
x=torch.tril(x)
z, weights, attention_score=attention(q,k,v,x)
print(z)

tensor([[-2.8869e-01, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [ 1.7191e+00, -1.6710e+00, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [-4.1048e-01, -3.4469e-01, -8.7687e-02, -1.0000e+09, -1.0000e+09],
        [ 2.4798e-01,  1.1101e+00,  7.8945e-01,  6.7183e-01, -1.0000e+09],
        [ 2.9106e+00, -7.4744e-01,  1.4711e+00, -3.3173e-01, -7.5192e-01]])


In [8]:
# Multi-Head Attention
import torch.nn as nn
import torch.nn.functional as F

class MHA(nn.Module):
    def __init__(self,heads,d_model):
        super().__init__()
        self.heads=heads
        self.d_model=d_model
        self.d_k=d_model//heads
        self.w_k=nn.Linear(d_model,d_model) #weights
        self.w_q=nn.Linear(d_model,d_model)
        self.w_v=nn.Linear(d_model,d_model)
        self.w_o=nn.Linear(d_model,d_model)
      
    def forward(self,query,key,value,mask=None):
        batchq=query.size(0)
        batchk=key.size(0)
        batchv=value.size(0)
        
        
        queries=self.w_q(query)
        keys=self.w_k(key)
        values=self.w_v(value)
        queries=queries.view(batchq,-1,self.heads,self.d_k).transpose(1,2)
        keys=keys.view(batchk,-1,self.heads,self.d_k).transpose(1,2)
        values=values.view(batchv,-1,self.heads,self.d_k).transpose(1,2)

        _,_, attention_score=attention(queries,keys,values)
   
        if mask is not None:
            _,_, attention_score=attention(queries,keys,values,mask)
        
        attention_score=attention_score.transpose(1,2).contiguous().view(batchq,-1,self.d_model)
        
        del query
        del key
        del value
        return self.w_o(attention_score)


In [9]:
x=torch.rand(size=(8,5,12))
mha=MHA(heads=6,d_model=x.shape[-1],)
output=mha(x,x,x)
print (output.shape)

torch.Size([8, 5, 12])


In [10]:

# Test 1: identical keys -> uniform attention -> output = mean(v)
seq_len = 5
d = 8

k_same = torch.randn(1, d).repeat(seq_len, 1)   # every row identical
v_test = torch.randn(size=(seq_len, d))
q_test = torch.randn(size=(seq_len, d))         # queries irrelevant when keys are identical

z_t, weights_t, out_t = attention(q_test,k_same, v_test, )

print("weights (each row should be ~1/seq_len):\n", weights_t)

expected = v_test.mean(dim=0)                   # hand-computed plain average of values
expected_full = expected.unsqueeze(0).expand(seq_len, -1)

print("expected (v.mean broadcast to all positions):\n", expected_full)
print("actual attention output:\n", out_t)
print("match:", torch.allclose(out_t, expected_full, atol=1e-5))

weights (each row should be ~1/seq_len):
 tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000]])
expected (v.mean broadcast to all positions):
 tensor([[-0.6592,  0.3478, -0.0795,  0.6694, -0.6288, -0.0394,  0.5525,  0.2104],
        [-0.6592,  0.3478, -0.0795,  0.6694, -0.6288, -0.0394,  0.5525,  0.2104],
        [-0.6592,  0.3478, -0.0795,  0.6694, -0.6288, -0.0394,  0.5525,  0.2104],
        [-0.6592,  0.3478, -0.0795,  0.6694, -0.6288, -0.0394,  0.5525,  0.2104],
        [-0.6592,  0.3478, -0.0795,  0.6694, -0.6288, -0.0394,  0.5525,  0.2104]])
actual attention output:
 tensor([[-0.6592,  0.3478, -0.0795,  0.6694, -0.6288, -0.0394,  0.5525,  0.2104],
        [-0.6592,  0.3478, -0.0795,  0.6694, -0.6288, -0.0394,  0.5525,  0.2104],
        [-0.6592,  0.3478, -0.0795,  0.6694, -0.6288, -0.03

In [11]:
# Test 2: causal mask -> no position attends to the future (strict upper triangle == 0)
mask = torch.tril(torch.ones(seq_len, seq_len))   # 1 = allowed, 0 = masked out

z_c, weights_c, out_c = attention(k, v, q, mask=mask)

print("weights with causal mask:\n", weights_c)

upper = torch.triu(weights_c, diagonal=1)         # everything above the diagonal
print("strict upper triangle (should be all zero):\n", upper)
print("all zero:", torch.allclose(upper, torch.zeros_like(upper)))

weights with causal mask:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7647, 0.2353, 0.0000, 0.0000, 0.0000],
        [0.2987, 0.6074, 0.0939, 0.0000, 0.0000],
        [0.1987, 0.0975, 0.4517, 0.2521, 0.0000],
        [0.0624, 0.2965, 0.3951, 0.0460, 0.2000]])
strict upper triangle (should be all zero):
 tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])
all zero: True


In [12]:
class FullyConnected(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.1):
        super(FullyConnected,self).__init__()
        self.layer_w1=nn.Linear(d_model,d_ff)
        self.layer_w2=nn.Linear(d_ff,d_model)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        return self.layer_w2(self.dropout(self.layer_w1(x).relu()))
        

In [13]:
"""
class residuals(nn.Module):
    def __init__(self,layer,size,dropout):
        super(residuals, self).__init__() 
        self.size=size
        self.layer=nn.LayerNorm(size)
        self.dropout=nn.Dropout(dropout)
    def forwards(self,x,heads):
        mha=MHA(heads,d_model=x.shape[-1],)  # restructure to include genral sublayer no t attention( move to __init__ then call as self.)
        output=self.layer(x+mha(x))
        return output
"""

'\nclass residuals(nn.Module):\n    def __init__(self,layer,size,dropout):\n        super(residuals, self).__init__() \n        self.size=size\n        self.layer=nn.LayerNorm(size)\n        self.dropout=nn.Dropout(dropout)\n    def forwards(self,x,heads):\n        mha=MHA(heads,d_model=x.shape[-1],)  # restructure to include genral sublayer no t attention( move to __init__ then call as self.)\n        output=self.layer(x+mha(x))\n        return output\n'

In [14]:
class SublayerConnection(nn.Module):
    def __init__(self,size,dropout):
        super(SublayerConnection, self).__init__()
        self.size=size
        self.NormLayer = nn.LayerNorm(size)
        self.dropout=nn.Dropout(dropout)
    def forward (self,x,sublayer):
        return x + self.dropout(sublayer(self.NormLayer(x)))
    

In [15]:
class EncoderLayer(nn.Module):
    def __init__(self, feedfwd,self_attn,size,dropout):
        super(EncoderLayer, self).__init__()
        self.size=size
        self.feedfwd=feedfwd
        self.self_attn=self_attn
        self.dropout=nn.Dropout(dropout)
        self.sublayer=clones(SublayerConnection(size,dropout),2)
    def forward(self,x,mask):
        x= self.sublayer[0](x,lambda x:self.self_attn(x,x,x,mask))
        return self.sublayer[1](x,self.feedfwd)


In [29]:
class Encoder (nn.Module):
    def __init__(self,layers,N):
        super(Encoder, self).__init__()
        self.N=N
        self.layers=clones(layers,N)
        self.NormLayer=nn.LayerNorm(layers.size)
    def forward(self,x,mask):
        for layer in self.layers:
            x=layer(x,mask)
        return self.NormLayer(x)
        

In [17]:
import copy 

def clones (module,N):
    """
    Produce N identical layers with learnable parameters
    """
    return nn.ModuleList([copy.deepcopy(module) for _ in range (N)])

In [30]:
#Test:
ffn = FullyConnected(d_model=512, d_ff=2048)
attn = MHA(heads=8, d_model=512)
layer = EncoderLayer(ffn, attn, size=512, dropout=0.1)
enc = Encoder(layer, N=6)
enc(torch.rand(2,10,512), mask=None).shape

torch.Size([2, 10, 512])

In [31]:
enc.layers[0] is enc.layers[1]

False

In [32]:
class DecoderLayer(nn.Module):
    def __init__(self, feedfwd,masked_attn,cross_attn,size,dropout):
        super(DecoderLayer, self).__init__()
        self.size=size
        self.feedfwd=feedfwd
        self.masked_attn=masked_attn
        self.cross_attn=cross_attn
        self.dropout=nn.Dropout(dropout)
        self.sublayer=clones(SublayerConnection(size,dropout),3)
    def forward(self,x,memory,src_mask,tgt_mask):
        x=self.sublayer[0](x, lambda x:self.masked_attn(x,x,x,tgt_mask))
        x=self.sublayer[1](x, lambda x:self.cross_attn(x,memory,memory,src_mask))
        return self.sublayer[2](x,self.feedfwd)

In [21]:
class Decoder (nn.Module):
    def __init__(self,layers,N):
        super(Decoder, self).__init__()
        self.N=N
        self.layers=clones(layers,N)
        self.NormLayer=nn.LayerNorm(layers.size)
    def forward(self,x,memory,src_mask,tgt_mask):
        for layer in self.layers:
            x=layer(x,memory,src_mask,tgt_mask)
        return self.NormLayer(x)
        

In [22]:
def subsequent_mask(size): #mask next position
    shape=(1,size,size)    #bc MHA's attention score ies in [batch,head, seq,seq]
    mask=torch.triu(torch.ones(shape),diagonal=1).type(torch.uint8)
    return mask==0

    

In [23]:
class Embedding(nn.Module):
    def __init__(self,vocab,d_model):
        super(Embedding, self).__init__()
        self.d_model=d_model
        self.embedding=nn.Embedding(vocab,d_model)
    def forward(self,x):
        return self.embedding(x)*math.sqrt(self.d_model)
        

In [33]:
class PE(nn.Module):
    def __init__(self,d_model,dropout,max_len=5000 ):
        super(PE, self).__init__()
        self.dropout=nn.Dropout(p=dropout)
        self.d_model=d_model
        pos=torch.arange(max_len).unsqueeze(1)  #build position column vector
        freq_scale=torch.exp(torch.arange(0,d_model,2)*-(math.log(10000.0)/d_model)) # log scale for practical numercical stability
        pe=torch.zeros(max_len,d_model)  # dimentional matrix
        pe[:,0::2]=torch.sin(pos*freq_scale)
        pe[:,1::2]=torch.cos(pos*freq_scale)
        pe=pe.unsqueeze(0)  # to add the batch dim [1, max_len, d_model]
        self.register_buffer("pe",pe) # to move it to the GPU; PE matrix not learbable to prevent device mismatch
        
    def forward(self,x):
        x=x+self.pe[:,: x.size(1)].requires_grad_(False)
        return self.dropout(x)
        
        
        
        
        
        
    

In [34]:
class EncoderDecoder(nn.Module):
    def __init__(self,encoder,decoder,src_embed,tgt_embed,generator):
        super(EncoderDecoder,self).__init__()
        self.encoder=encoder
        self.decoder=decoder
        self.generator=generator
        self.src_embed=src_embed
        self.tgt_embed=tgt_embed
    
    def forward (self,src,tgt,src_mask,tgt_mask):
        return self.decode(self.encode(src,src_mask),src_mask,tgt,tgt_mask) # returns decoder hidden states
    def encode (self,src,src_mask):
        return self.encoder(self.src_embed(src),src_mask) #returns memory
    def decode (self,memory,src_mask,tgt,tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)   

    

In [35]:
from torch.nn.functional import log_softmax

class Generator(nn.Module):
    def __init__(self,d_model,vocab):
        super(Generator,self).__init__()
        self.linear=nn.Linear(d_model,vocab)
    def forward(self,x):
        return log_softmax(self.linear(x),dim=-1)

In [36]:
def make_model(src_vocab,tgt_vocab,N=6,d_model=512,d_ff=2048,h=8,dropout=0.1):
    c=copy.deepcopy # deep copy for attn and ff to NOT share weights
    attn=MHA(h,d_model)
    ff=FullyConnected(d_model,d_ff,dropout)
    position=PE(d_model,dropout)
    model=EncoderDecoder(Encoder(EncoderLayer(c(ff),c(attn),d_model,dropout),N),
                        Decoder(DecoderLayer(c(ff),c(attn),c(attn),d_model,dropout),N),
                        nn.Sequential(Embedding(src_vocab,d_model),c(position)),
                        nn.Sequential(Embedding(tgt_vocab,d_model),c(position)),
                        Generator(d_model,tgt_vocab),
                                )    
    
    # This was important from their code.
    # Initialize parameters with Glorot / fan_avg.
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model

In [37]:
"""
Copy-task overfit test for the from-scratch Transformer.

The model is trained to reproduce its input sequence. If every component
(masking, residuals, cross-attention, init, generator) is wired correctly,
loss drops to near zero within a few hundred steps.

ASSUMED INTERFACE:
    make_model(src_vocab, tgt_vocab, N, d_model, d_ff, h, dropout) -> EncoderDecoder
    model(src, tgt, src_mask, tgt_mask)      -> [batch, seq, d_model]
    model.generator(x)                       -> log-probs [batch, seq, vocab]
    model.encode(src, src_mask)              -> memory
    model.decode(memory, src_mask, tgt, tgt_mask)
    subsequent_mask(size)                    -> lower-triangular, 1 = allowed

MASK SHAPES: this harness emits 4-D masks ([b, 1, 1, s] and [b, 1, t, t]) so they
broadcast directly against your [batch, heads, seq, seq] attention scores. If your
MHA already unsqueezes internally, drop the _to_4d calls.
"""

import time
import torch
import torch.nn as nn


def _to_4d(mask):
    """Pad a mask with dims (after the batch dim) until it is [b, *, s, s] rank-4."""
    while mask.dim() < 4:
        mask = mask.unsqueeze(1)
    return mask


class Batch:
    """One training example pair plus the masks needed for it."""

    def __init__(self, src, tgt=None, pad=0):
        self.src = src
        self.src_mask = _to_4d((src != pad).unsqueeze(-2))          # [b, 1, 1, src_len]
        if tgt is not None:
            self.tgt = tgt[:, :-1]                                  # decoder input
            self.tgt_y = tgt[:, 1:]                                 # expected output
            self.tgt_mask = self.make_std_mask(self.tgt, pad)
            self.ntokens = (self.tgt_y != pad).data.sum()

    @staticmethod
    def make_std_mask(tgt, pad):
        pad_mask = _to_4d((tgt != pad).unsqueeze(-2))               # [b, 1, 1, t]
        causal = _to_4d(subsequent_mask(tgt.size(-1))).type_as(pad_mask)
        return pad_mask & causal                                    # [b, 1, t, t]


def data_gen(vocab, batch_size, nbatches, seq_len=10, device="cpu"):
    """Random integer sequences where target == source."""
    for _ in range(nbatches):
        data = torch.randint(1, vocab, size=(batch_size, seq_len), device=device)
        data[:, 0] = 1                                              # start token
        src = data.clone().detach()
        tgt = data.clone().detach()
        yield Batch(src, tgt, pad=0)


class SimpleLossCompute:
    """Apply the generator, compute NLL loss, backprop, step."""

    def __init__(self, generator, criterion, optimizer=None):
        self.generator = generator
        self.criterion = criterion
        self.optimizer = optimizer

    def __call__(self, x, y, norm):
        x = self.generator(x)
        loss = self.criterion(
            x.contiguous().view(-1, x.size(-1)),
            y.contiguous().view(-1),
        ) / norm
        if self.optimizer is not None:
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad(set_to_none=True)
        return loss.item() * norm


def run_epoch(data_iter, model, loss_compute, log_every=20):
    start = time.time()
    total_tokens, total_loss, tokens = 0, 0.0, 0
    for i, batch in enumerate(data_iter):
        out = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
        loss = loss_compute(out, batch.tgt_y, batch.ntokens)
        total_loss += loss
        total_tokens += batch.ntokens
        tokens += batch.ntokens
        if log_every and i % log_every == 1:
            elapsed = time.time() - start
            print(f"  step {i:4d} | loss/token {loss / batch.ntokens:7.4f} "
                  f"| tok/s {tokens / max(elapsed, 1e-9):7.1f}")
            start, tokens = time.time(), 0
    return total_loss / total_tokens


@torch.no_grad()
def greedy_decode(model, src, src_mask, max_len, start_symbol):
    model.eval()
    memory = model.encode(src, src_mask)
    ys = torch.zeros(1, 1).fill_(start_symbol).type_as(src)
    for _ in range(max_len - 1):
        tgt_mask = _to_4d(subsequent_mask(ys.size(1))).type_as(src) > 0
        out = model.decode(memory, src_mask, ys, tgt_mask)
        prob = model.generator(out[:, -1])
        next_word = torch.argmax(prob, dim=-1).item()
        ys = torch.cat([ys, torch.zeros(1, 1).type_as(src).fill_(next_word)], dim=1)
    return ys


def main():
    torch.manual_seed(0)
    V, SEQ_LEN = 11, 10
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = make_model(V, V, N=2, d_model=64, d_ff=256, h=4, dropout=0.1).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"device={device}  parameters={n_params:,}\n")

    criterion = nn.NLLLoss(reduction="sum", ignore_index=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)

    for epoch in range(10):
        model.train()
        print(f"epoch {epoch}")
        run_epoch(
            data_gen(V, batch_size=32, nbatches=40, seq_len=SEQ_LEN, device=device),
            model,
            SimpleLossCompute(model.generator, criterion, optimizer),
        )

    print("\n--- greedy decode check ---")
    src = torch.randint(1, V, size=(1, SEQ_LEN), device=device)
    src[:, 0] = 1
    src_mask = torch.ones(1, 1, 1, SEQ_LEN, device=device) > 0
    out = greedy_decode(model, src, src_mask, max_len=SEQ_LEN, start_symbol=1)
    print("source :", src.tolist()[0])
    print("decoded:", out.tolist()[0])
    match = sum(int(a == b) for a, b in zip(src.tolist()[0][1:], out.tolist()[0][1:]))
    print(f"matched {match}/{SEQ_LEN - 1} positions after the start token")


if __name__ == "__main__":
    main()

device=cuda  parameters=235,851

epoch 0
  step    1 | loss/token  2.9617 | tok/s   245.5
  step   21 | loss/token  2.0649 | tok/s  4439.9
epoch 1
  step    1 | loss/token  1.8465 | tok/s  5174.2
  step   21 | loss/token  1.7854 | tok/s  4746.6
epoch 2
  step    1 | loss/token  1.6077 | tok/s  4900.2
  step   21 | loss/token  1.4617 | tok/s  5242.3
epoch 3
  step    1 | loss/token  1.2828 | tok/s  5050.8
  step   21 | loss/token  1.1338 | tok/s  5398.3
epoch 4
  step    1 | loss/token  1.0387 | tok/s  5167.9
  step   21 | loss/token  0.8791 | tok/s  5309.2
epoch 5
  step    1 | loss/token  0.6483 | tok/s  4618.9
  step   21 | loss/token  0.6966 | tok/s  5224.6
epoch 6
  step    1 | loss/token  0.5727 | tok/s  5764.1
  step   21 | loss/token  0.5163 | tok/s  5162.7
epoch 7
  step    1 | loss/token  0.5211 | tok/s  5529.6
  step   21 | loss/token  0.3577 | tok/s  5199.1
epoch 8
  step    1 | loss/token  0.3482 | tok/s  5216.1
  step   21 | loss/token  0.3078 | tok/s  5341.6
epoch 9
  ste